In [16]:
import sys
import os

# Adiciona o diretório pai (a raiz do projeto) ao 'sys.path'
# '..' significa "subir um nível"
project_root = os.path.abspath('..')

# Adiciona o caminho apenas se ele ainda não estiver lá
if project_root not in sys.path:
    sys.path.append(project_root)

# Agora esta linha deve funcionar!
from config import settings
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

settings.PIORES_RANKINGS_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.float_format = '{:.2f}'.format

# Leitura dos Dados

In [6]:
df_insta = pd.read_excel(settings.DEPUTADOS_INSTA_XLSX_OUT)

df_insta.columns

Index(['Nome Parlamentar', 'Partido', 'Telefone', 'Correio Eletrônico',
       'Nome sem Acento', 'Tratamento', 'Nome Civil', 'Instagram'],
      dtype='str')

In [7]:
# Ler o Excel em um DataFrame
df_contatos = pd.read_excel(settings.DEPUTADOS_CONTATOS_XLSX_IN)

df_contatos.columns

Index(['Nome Civil', 'Partido', 'E-mail', 'Telefone', 'Endereço',
       'Data de Nascimento', 'Naturalidade'],
      dtype='str')

In [8]:
# 2. Limpar espaços em branco nos nomes para garantir o cruzamento correto
df_insta['Nome Civil'] = df_insta['Nome Civil'].str.strip()
df_contatos['Nome Civil'] = df_contatos['Nome Civil'].str.strip()

# 3. Selecionar apenas as colunas necessárias da planilha que tem o Instagram
# Usamos 'Nome Civil' para o cruzamento e 'Instagram' como o dado que queremos trazer
df_subset_insta = df_insta[['Nome Civil', 'Instagram']].drop_duplicates(subset='Nome Civil')

# 4. Realizar o merge (união) do tipo 'left'
# Isso mantém todos os dados da planilha de contatos e adiciona o Instagram onde houver correspondência
df_contatos_insta = pd.merge(df_contatos, df_subset_insta, on='Nome Civil', how='left')

df_contatos_insta.columns

Index(['Nome Civil', 'Partido', 'E-mail', 'Telefone', 'Endereço',
       'Data de Nascimento', 'Naturalidade', 'Instagram'],
      dtype='str')

In [9]:
dados_dict = {}

# 1. Usar 'with open' para abrir e fechar o arquivo automaticamente
# 'r' significa modo de leitura (read)
try:
    with open(settings.APIFY_JSON_OUT, 'r', encoding='utf-8') as arquivo:
        # 2. Usar json.load() para ler o arquivo e converter para dict
        dados_dict = json.load(arquivo)

    df_deputados = pd.read_json(settings.APIFY_JSON_OUT, orient='records')

except FileNotFoundError:
    print(f"Erro: O arquivo '{settings.APIFY_CSV_OUT}' não foi encontrado.")
except json.JSONDecodeError:
    print(f"Erro: O arquivo '{settings.APIFY_CSV_OUT}' não é um JSON válido.")

In [10]:
list_of_post_dataframes = []

for registro in dados_dict:
    # Check if the key exists AND if the value is not empty (for safety)
    if 'latestPosts' in registro and registro['latestPosts']:
        
        posts = registro['latestPosts']
        
        for post in posts:
            post['inputUrl'] = registro['inputUrl']
        
        df_posts = pd.DataFrame(posts)
        list_of_post_dataframes.append(df_posts)

# 1. Concatenate all DataFrames in the list into one master DataFrame
# 'ignore_index=True' is used to reset the index of the resulting DataFrame
df_posts_total = pd.concat(list_of_post_dataframes, ignore_index=True)

# 2. Convert the 'timestamp' column to datetime objects
# Assuming 'timestamp' is in seconds (a common format for API timestamps)
df_posts_total['data_datetime'] = pd.to_datetime(df_posts_total['timestamp'])

In [11]:
df_posts_agrupado = df_posts_total.groupby('inputUrl').agg(
    commentsCount=('commentsCount', 'sum'),
    likesCount=('likesCount', 'sum'),
    videoViewCount=('videoViewCount', 'sum'),
    post_count=('id', 'count'),
    data_datetime_max=('data_datetime', 'max'),
    data_datetime_min=('data_datetime', 'min'),
).reset_index()

df_deputados_join = pd.merge(df_deputados, df_posts_agrupado, how='left', on='inputUrl')

df_deputados_join['% Engajamento'] = (df_deputados_join['likesCount'] + df_deputados_join['commentsCount']) / df_deputados_join['followersCount']

df_deputados_join['% commentsCount'] = df_deputados_join['commentsCount'] / df_deputados_join['followersCount']

df_deputados_join['% likesCount'] = df_deputados_join['likesCount'] / df_deputados_join['followersCount']

df_deputados_join['likesCount\Posts'] = df_deputados_join['likesCount'] / df_deputados_join['post_count']

df_deputados_join['commentsCount\Posts'] = df_deputados_join['commentsCount'] / df_deputados_join['post_count']

# 1. Calcular o período em dias que os posts de cada deputado cobrem
periodo_dias = (df_deputados_join['data_datetime_max'] - df_deputados_join['data_datetime_min']).dt.days

# 2. Calcular a frequência (ex: posts por dia)
# Adicionamos +1 para evitar divisão por zero se todos os posts foram no mesmo dia
df_deputados_join['Frequencia (Posts/Dia)'] = df_deputados_join['post_count'] / (periodo_dias + 1)

# Ou, se preferir (dias por post):
df_deputados_join['Frequencia (Dias/Post)'] = periodo_dias / df_deputados_join['post_count']

<>:18: SyntaxWarning: invalid escape sequence '\P'
<>:20: SyntaxWarning: invalid escape sequence '\P'
<>:18: SyntaxWarning: invalid escape sequence '\P'
<>:20: SyntaxWarning: invalid escape sequence '\P'
C:\Users\vinic\AppData\Local\Temp\ipykernel_20716\1071656010.py:18: SyntaxWarning: invalid escape sequence '\P'
  df_deputados_join['likesCount\Posts'] = df_deputados_join['likesCount'] / df_deputados_join['post_count']
C:\Users\vinic\AppData\Local\Temp\ipykernel_20716\1071656010.py:20: SyntaxWarning: invalid escape sequence '\P'
  df_deputados_join['commentsCount\Posts'] = df_deputados_join['commentsCount'] / df_deputados_join['post_count']


In [12]:
df_deputados.columns

Index(['inputUrl', 'id', 'username', 'url', 'fullName', 'biography',
       'externalUrls', 'externalUrl', 'externalUrlShimmed', 'followersCount',
       'followsCount', 'hasChannel', 'highlightReelCount', 'isBusinessAccount',
       'joinedRecently', 'businessCategoryName', 'private', 'verified',
       'profilePicUrl', 'profilePicUrlHD', 'igtvVideoCount', 'relatedProfiles',
       'latestIgtvVideos', 'postsCount', 'latestPosts', 'fbid',
       'businessAddress', 'error', 'errorDescription', 'isRestrictedProfile',
       'restrictionReason'],
      dtype='str')

In [13]:
df_posts_total.columns

Index(['id', 'type', 'shortCode', 'caption', 'hashtags', 'mentions', 'url',
       'commentsCount', 'dimensionsHeight', 'dimensionsWidth', 'displayUrl',
       'images', 'videoUrl', 'alt', 'likesCount', 'videoViewCount',
       'timestamp', 'childPosts', 'ownerUsername', 'ownerId', 'productType',
       'isCommentsDisabled', 'inputUrl', 'taggedUsers', 'locationName',
       'locationId', 'musicInfo', 'isPinned', 'data_datetime'],
      dtype='str')

In [14]:
df_deputados_join.columns

Index(['inputUrl', 'id', 'username', 'url', 'fullName', 'biography',
       'externalUrls', 'externalUrl', 'externalUrlShimmed', 'followersCount',
       'followsCount', 'hasChannel', 'highlightReelCount', 'isBusinessAccount',
       'joinedRecently', 'businessCategoryName', 'private', 'verified',
       'profilePicUrl', 'profilePicUrlHD', 'igtvVideoCount', 'relatedProfiles',
       'latestIgtvVideos', 'postsCount', 'latestPosts', 'fbid',
       'businessAddress', 'error', 'errorDescription', 'isRestrictedProfile',
       'restrictionReason', 'commentsCount', 'likesCount', 'videoViewCount',
       'post_count', 'data_datetime_max', 'data_datetime_min', '% Engajamento',
       '% commentsCount', '% likesCount', 'likesCount\Posts',
       'commentsCount\Posts', 'Frequencia (Posts/Dia)',
       'Frequencia (Dias/Post)'],
      dtype='str')

# Segmentação

In [15]:
metrics = [
    '% Engajamento',
    '% likesCount',
    '% commentsCount',
    'likesCount\\Posts',
    'commentsCount\\Posts',
    'Frequencia (Posts/Dia)',
    'followersCount',
]

for metric in metrics:
    
    # Filtrar e ordenar como no código original
    df_piores = (
        df_deputados_join[df_deputados_join[metric].notna()]
        .sort_values(metric, ascending=True)
        .head(25)
    )
    
    print(f"Piores 10 para a métrica '{metric}':")
    print(df_piores[['inputUrl', metric]])
    print()

    set_piores_inputUrl = set()

    # Adicionar os inputUrl ao set
    set_piores_inputUrl.update(df_piores['inputUrl'].dropna())

    # Filtrar o DataFrame do Excel
    df_contatos_filtrados = df_contatos_insta[df_contatos_insta['Instagram'].isin(set_piores_inputUrl)]

    safe_metric = metric.replace('\\', '-').replace('/', '-')
    output_path = settings.PIORES_RANKINGS_DIR / f'{safe_metric}.xlsx'

    df_contatos_filtrados.to_excel(output_path, index=False)
    print(f"Arquivo salvo em: {output_path}")

Piores 10 para a métrica '% Engajamento':
                                              inputUrl  % Engajamento
119                https://www.instagram.com/leoprates           0.01
210               https://www.instagram.com/vitorlippi           0.01
194        https://www.instagram.com/pompeodemattospdt           0.02
214  https://www.instagram.com/viniciuscarvalhooficial           0.02
261              https://www.instagram.com/silvyealves           0.02
191        https://www.instagram.com/renilcenicodemoss           0.02
103        https://www.instagram.com/coronelfernandamt           0.02
75      https://www.instagram.com/fredlinharesbrasilia           0.02
9           https://www.instagram.com/dep_albertofraga           0.02
273       https://www.instagram.com/cezinhademadureira           0.02
184  https://www.instagram.com/professoralcidesoficial           0.02
196        https://www.instagram.com/romerorodriguespb           0.02
139              https://www.instagram.com/marci